# 04 - RAG Chatbot

This notebook builds the final RAG-based cardiovascular chatbot.

RAG pipeline:
1. Load raw cardiovascular documents
2. Split documents into chunks
3. Retrieve relevant chunks using TF-IDF
4. Insert retrieved context into the prompt
5. Generate grounded answers using the fine-tuned Qwen LoRA model

In [3]:
import json
import re
from pathlib import Path

import torch
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [4]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = MODEL_DIR / "qwen2_5_1_5b_cardio_lora"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("LoRA path:", LORA_PATH)
print("Device:", device)

Base directory: d:\CardioBot_NLP_Final
Raw directory: d:\CardioBot_NLP_Final\data\raw
LoRA path: d:\CardioBot_NLP_Final\models\qwen2_5_1_5b_cardio_lora
Device: cuda


In [5]:
raw_files = list(RAW_DIR.glob("*.txt"))

documents = []

for file_path in raw_files:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    documents.append({
        "source": file_path.name,
        "text": text
    })

print("Total raw documents:", len(documents))

for doc in documents[:5]:
    print("-", doc["source"])

Total raw documents: 29
- Angiography.txt
- Angioplasty and stent.txt
- Arrhythmia.txt
- Atherosclerosis.txt
- Blood pressure measurement.txt


In [6]:
def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Cleaning completed.")
print(documents[0]["text"][:500])

Cleaning completed.
Angiography
Definition
Angiography is a type of X-ray imaging used to examine blood vessels and assess blood flow. Because blood vessels do not appear clearly on a normal X-ray, a special dye called a contrast agent is injected into the bloodstream to make them visible. This allows doctors to detect abnormalities or blockages. The images produced during this procedure are called angiograms.
Purpose
Angiography is used to evaluate the condition of blood vessels and how blood flows through them. I


In [7]:
def chunk_text(documents, chunk_size=180, overlap=30):
    chunks = []

    for doc in documents:
        paragraphs = [p.strip() for p in doc["text"].split("\n") if p.strip()]

        for para in paragraphs:
            if len(para.split()) < 20:
                continue

            sentences = re.split(r'(?<=[.!?])\s+', para)

            current_chunk = []
            current_length = 0

            for sentence in sentences:
                words = sentence.split()

                if current_length + len(words) <= chunk_size:
                    current_chunk.append(sentence)
                    current_length += len(words)
                else:
                    chunk_str = " ".join(current_chunk).strip()

                    if len(chunk_str.split()) >= 30:
                        chunks.append({
                            "source": doc["source"],
                            "text": chunk_str
                        })

                    current_chunk = current_chunk[-1:]
                    current_length = sum(len(s.split()) for s in current_chunk)

                    current_chunk.append(sentence)
                    current_length += len(words)

            if current_chunk:
                chunk_str = " ".join(current_chunk).strip()

                if len(chunk_str.split()) >= 30:
                    chunks.append({
                        "source": doc["source"],
                        "text": chunk_str
                    })

    unique_chunks = []
    seen = set()

    for chunk in chunks:
        if chunk["text"] not in seen:
            unique_chunks.append(chunk)
            seen.add(chunk["text"])

    return unique_chunks


chunks = chunk_text(documents)

print("Total chunks:", len(chunks))
pd.DataFrame(chunks).head()

Total chunks: 303


,source,text
0,Angiography.txt,Angiography is a type of X-ray imaging used to...
1,Angiography.txt,Angiography is used to evaluate the condition ...
2,Angiography.txt,Angiography is usually performed in a hospital...
3,Angiography.txt,Angiography is generally considered safe and p...
4,Angiography.txt,There are several types of angiography dependi...


In [8]:
texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

tfidf_matrix = vectorizer.fit_transform(texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (303, 7613)


In [9]:
def retrieve_context(question, top_k=3):
    query_vec = vectorizer.transform([question])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(scores[idx])
        })

    return results

In [10]:
question = "What are the symptoms of stroke?"

retrieved = retrieve_context(question, top_k=3)

for i, item in enumerate(retrieved, start=1):
    print(f"Rank {i}")
    print("Source:", item["source"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:500])
    print("-" * 80)

Rank 1
Source: Stroke.txt
Score: 0.2608
Call 911 (or your local emergency services number) if you think you’re experiencing stroke symptoms again. Another stroke has an even higher risk of causing severe complications and being fatal. Don’t wait to call for help or go to the emergency room.
--------------------------------------------------------------------------------
Rank 2
Source: Stroke.txt
Score: 0.1254
Stroke rehab is an important part of stroke treatment. You’ll need rehab to help you adjust to changes in your brain and body after a stroke. You may need to regain abilities you had before or adjust to new or different disabilities. You might need a combination of:
--------------------------------------------------------------------------------
Rank 3
Source: Stroke.txt
Score: 0.1039
Visit a healthcare provider for a check-up every year (or as often as they suggest). Many of the health conditions and issues that can cause stroke develop and build up over time, and may not cause s

In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch_dtype = torch.float16 if device == "cuda" else torch.float32

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
    device_map="auto" if device == "cuda" else None
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

print("Fine-tuned LoRA model loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned LoRA model loaded.


In [12]:
SYSTEM_PROMPT = (
    "You are CardioBot, a cardiovascular health education assistant. "
    "Answer only using the provided context. "
    "If the answer is not available in the context, say that the information is not available in the knowledge base. "
    "Do not provide diagnosis, prescriptions, or emergency medical decisions. "
    "For emergency symptoms, advise the user to seek immediate medical help."
)


def build_rag_prompt(question, retrieved_contexts):
    context_text = ""

    for i, item in enumerate(retrieved_contexts, start=1):
        context_text += f"[Context {i} | Source: {item['source']}]\n"
        context_text += item["text"] + "\n\n"

    user_prompt = (
        f"Context:\n{context_text}\n"
        f"Question: {question}\n\n"
        f"Answer clearly and completely based only on the context. If the question asks for a process or flow, explain the steps in order from beginning to end."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [13]:
def generate_rag_answer(question, top_k=3, max_new_tokens=300, min_score=0.05):
    retrieved = retrieve_context(question, top_k=top_k)

    if retrieved[0]["score"] < min_score:
        return {
            "answer": "The information is not available in the cardiovascular knowledge base.",
            "retrieved_contexts": retrieved
        }

    prompt = build_rag_prompt(question, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.15,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "answer": answer,
        "retrieved_contexts": retrieved
    }

In [14]:
question = "What are the symptoms of cardiomyopathy?"

result = generate_rag_answer(question)

print("Question:")
print(question)

print("\nRAG Answer:")
print(result["answer"])

print("\nRetrieved Sources:")
for item in result["retrieved_contexts"]:
    print("-", item["source"], "| score:", round(item["score"], 4))

Question:
What are the symptoms of cardiomyopathy?

RAG Answer:
The symptoms of cardiomyopathy may include shortness of breath, fatigue, dizziness, fainting, swelling in the legs, ankles, feet, abdomen, or around the eyes, and chest pain.

Retrieved Sources:
- Cardiomyopathy.txt | score: 0.3588
- Cardiomyopathy.txt | score: 0.3055
- Cardiomyopathy.txt | score: 0.2718


In [15]:
questions = [
    "What is a pacemaker?",
    "What are the warning signs of stroke?",
    "How is hypertension diagnosed?",
    "What is cardiac rehabilitation?"
]

for q in questions:
    result = generate_rag_answer(q)

    print("=" * 100)
    print("Question:", q)
    print("\nAnswer:")
    print(result["answer"])
    print("\nSources:")
    for item in result["retrieved_contexts"]:
        print("-", item["source"], "|", round(item["score"], 4))

Question: What is a pacemaker?

Answer:
A pacemaker is a small, battery-powered device that prevents the heart from beating too slowly. It is usually implanted under the skin near the collarbone during surgery.

Sources:
- Pacemaker.txt | 0.2576
- Pacemaker.txt | 0.2286
- Pacemaker.txt | 0.226
Question: What are the warning signs of stroke?

Answer:
Visit a healthcare provider for a check-up every year (or as often as they suggest). Many of the health conditions and issues that can cause stroke develop and build up over time, and may not cause symptoms you can notice. Many people with high blood pressure never feel or sense anything wrong. Your provider will help you catch and manage any warning signs before they increase your risk of a stroke later on.

Sources:
- Blood pressure measurement.txt | 0.2918
- Stroke.txt | 0.2583
- warning_signs_of_heart_attack.txt | 0.2106
Question: How is hypertension diagnosed?

Answer:
Hypertension is diagnosed by measuring blood pressure using a sphyg

In [16]:
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


test_data = read_jsonl(PROCESSED_DIR / "test.jsonl")

print("Test size:", len(test_data))
pd.DataFrame(test_data).head()

Test size: 34


,id,topic,source,question,answer
0,qa_047,Cardiomyopathy,Cardiomyopathy.txt,What is dilated cardiomyopathy?,Dilated cardiomyopathy is a type of cardiomyop...
1,qa_057,Heart Valve Disease,Heart Valve Disease.txt,What is valve regurgitation?,Valve regurgitation happens when valve flaps d...
2,qa_099,Blood Flow,Blood_Flow.txt,What are the main functions of blood flow?,Blood flow delivers oxygen and nutrients to or...
3,qa_135,Heart Failure,heart_failure.txt,What are the ACC/AHA stages of heart failure?,The ACC/AHA stages of heart failure are Stage ...
4,qa_070,Stroke,Stroke.txt,How is hemorrhagic stroke treated?,Hemorrhagic stroke treatment focuses on contro...


In [17]:
rag_results = []

for i, item in enumerate(test_data, start=1):
    question = item["question"]
    
    result = generate_rag_answer(question, top_k=3)

    rag_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "source": item["source"],
        "question": question,
        "reference_answer": item["answer"],
        "rag_answer": result["answer"],
        "top_source": result["retrieved_contexts"][0]["source"],
        "top_score": result["retrieved_contexts"][0]["score"],
        "retrieved_contexts": result["retrieved_contexts"]
    })

    print(f"[{i}/{len(test_data)}] {question}")
    print(result["answer"][:250])
    print("-" * 80)

[1/34] What is dilated cardiomyopathy?
Dilated cardiomyopathy occurs when the heart's chambers thin and stretch, growing larger. It typically starts in the left ventricle and can affect the ability to pump blood effectively.
--------------------------------------------------------------------------------
[2/34] What is valve regurgitation?
Valve regurgitation occurs when the valve flaps don't close properly. Blood may flow backward through the valve during each heartbeat cycle.
--------------------------------------------------------------------------------
[3/34] What are the main functions of blood flow?
The main functions of blood flow include delivering oxygen and nutrients to organs and tissues, removing carbon dioxide and metabolic waste, and transporting immune cells that protect the body from infections.
--------------------------------------------------------------------------------
[4/34] What are the ACC/AHA stages of heart failure?
The ACC/AHA stages of heart failure are St

In [18]:
rag_df = pd.DataFrame(rag_results)

rag_df["source_match"] = rag_df["source"] == rag_df["top_source"]

def top3_source_match(row):
    retrieved_sources = [ctx["source"] for ctx in row["retrieved_contexts"]]
    return row["source"] in retrieved_sources

rag_df["top3_source_match"] = rag_df.apply(top3_source_match, axis=1)

rag_summary = {
    "model": "RAG + Qwen LoRA",
    "test_size": len(rag_df),
    "source_match_accuracy": round(float(rag_df["source_match"].mean()), 4),
    "top3_source_match_accuracy": round(float(rag_df["top3_source_match"].mean()), 4),
    "average_top_score": round(float(rag_df["top_score"].mean()), 4)
}

rag_summary

{'model': 'RAG + Qwen LoRA',
 'test_size': 34,
 'source_match_accuracy': 0.6176,
 'top3_source_match_accuracy': 0.8529,
 'average_top_score': 0.2919}

In [19]:
with open(RESULTS_DIR / "rag_summary.json", "w", encoding="utf-8") as f:
    json.dump(rag_summary, f, indent=2, ensure_ascii=False)

print("Saved RAG summary.")

Saved RAG summary.


In [20]:
rag_output_path = RESULTS_DIR / "rag_answers.json"

with open(rag_output_path, "w", encoding="utf-8") as f:
    json.dump(rag_results, f, indent=2, ensure_ascii=False)

pd.DataFrame(rag_results).to_csv(RESULTS_DIR / "rag_results.csv", index=False)

print("Saved RAG results to:")
print(rag_output_path)
print(RESULTS_DIR / "rag_results.csv")

Saved RAG results to:
d:\CardioBot_NLP_Final\results\rag_answers.json
d:\CardioBot_NLP_Final\results\rag_results.csv
